# Geolocated Venues

## Contents

- Inspecting the Data Sets
  - Data Set 1
  - Data Set 2
  - Data Set 3
  - Data Set 4
- Evaluating Data Reliability
  - Evaluating Mapbox Data
  - Evaluating Tidyverse Data
  - Evaluating Student Data
    - Student Address Data
    - Student Zip Code Data
- Generate Complete List of Venue Locations
  - Student Zip Data
  - Tidyverse Data
  - Mapbox Data
  - Student Address Data
- Filling in the Blanks
- Quality Control
- Write the Geolocation Data to the Excel File
  
## Introduction

The next step in organizing the data is to gather together all the geolocations we have for the venues. This is a complicated task because there are numerous different sources of locations, gathered by different people over time. 

- Locations obtained by Information Specialist Diane Lopez
- Locations obtained by Information Specialist Javier Ruedas through the Mapbox API
- Locations obtained by a student worker supervised by Diane Lopez
- Locations obtained by Information Specialist Angelina Cortes through the R Tidyverse library

These different geolocation data sets have varying reliability due to the different APIs and research processes used to obtain them. I'll retrieve and inspect each data set in the sequence it was produced.

## Inspecting the Data Sets

### Data Set 1

Chronologically, the earliest venue geolocation data set was generated by Diane Lopez. This started with some address data provided by a faculty member to two librarians, in a spreadsheet called `Layer 1 Venues`. Diane Lopez supplemented this with her own resear h as well as queries to ArcGIS and Open Street Map APIs. She then combined this data with the concert history from `Venue_Artist_Event` into a spreadsheet called `Ven_Ev_Ar_Address`. I load the sheet and inspect it. There are 312 sets of coordinates associated with venues in this sheet. However, all the venues had some sort of address information.

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
original_data_sheet_names = pd.ExcelFile('input-data/product1_data_master.xlsx').sheet_names
original_data_sheet_names

['Ven_Ev_Ar_Address',
 'Venue_Artist_Event',
 'Layer 1 Venues',
 'artistsGenre',
 'genreIndex',
 'Sheet5',
 'artistsIndex',
 'Venues No Address']

In [3]:
layer_1_venues_orig = pd.read_excel('input-data/product1_data_master.xlsx', sheet_name='Layer 1 Venues')
layer_1_venues_orig.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Venue_Name  974 non-null    str    
 1   Longitude   312 non-null    float64
 2   Latitude    312 non-null    float64
 3   Address     974 non-null    str    
 4   Zip         916 non-null    float64
 5   City        912 non-null    str    
 6   State       972 non-null    str    
dtypes: float64(3), str(4)
memory usage: 53.4 KB


### Data Set 2

In March 2024, the I queried the Mapbox API to obtain coordinates for the data visualization. The data was stored in a JSON file called `venues.json`. Inspecting this data now, I see 903 venues had coordinate data.

Unfortunately, work focused entirely on getting the complete set of coordinates for `Layer 1 Venues`, without realizing that most of the venues that had concert data associated with them were not in `Layer 1 Venues`.

When I generated this data, I retained all geolocations previously obtained by Diane Lopez, so although I refer to this data set from now on as "Mapbox Data", it includes a lot of data obtained from ArcGIS and Open Stret Map as well as hands-on research by Diane Lopez.

In [4]:
mapbox_data = pd.read_json('input-data/venues.json')

In [5]:
mapbox_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         974 non-null    int64  
 1   name       974 non-null    str    
 2   address    974 non-null    str    
 3   city       912 non-null    str    
 4   state      972 non-null    str    
 5   zip        916 non-null    float64
 6   longitude  903 non-null    float64
 7   latitude   903 non-null    float64
dtypes: float64(3), int64(1), str(4)
memory usage: 61.0 KB


### Data Set 3

The third data set was obtained by a student worker under Diane Lopez' supervision. This was an effort to complete the data in `Layer 1 Venues`. In June 2025, Diane Lopez reached out to me to request a data analysis on missing venues. I analyzed the `Layer 1 Venues` data and shared the following Excel spreadsheets:
- `Missing Coordinates`, with 71 venues that had no coordinates
- `>30 Miles Away`, with 66 venues that were more than 30 miles away according to their coordinates, and I thus suspected to be bad data resulting from the Mapbox API query
- `Missing or Bad Address`, 36 venues having addresses either completely missing, or missing one or more parts
- `Missing Zip`, with 58 venues that were missing zip codes.

All this information was bundled in an Excel document called `venues_separated.xlsx` and passed on to Diane to share with the student worker. The student worker did some work on the bad addresses and zip codes, which was shared as two CSV files based on corresponding PDFs.

- `venues_separated MISSING AND BAD ADDRESSES.pdf` --> `geocode_results_from_missing_bad_address.xlsx`
- `venues_separated ZIP.pdf` --> `geocode_results_bad_zips.xlsx`

The pdf files have some useful comments added by the student worker regarding information she found in newspaper archives while doing research. The Excel files have, additionally, a `points` column with new coordinate data. 

In [6]:
student_address_data = pd.read_excel('input-data/geocode_results_from_missing_bad_address.xlsx')

In [7]:
student_address_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                36 non-null     int64  
 1   name              36 non-null     str    
 2   address           36 non-null     str    
 3   city              36 non-null     str    
 4   state             36 non-null     str    
 5   zip               36 non-null     int64  
 6   longitude         33 non-null     float64
 7   latitude          33 non-null     float64
 8   complete_address  36 non-null     str    
 9   location          21 non-null     str    
 10  point             21 non-null     str    
dtypes: float64(2), int64(2), str(7)
memory usage: 3.2 KB


In [8]:
student_zip_data = pd.read_excel('input-data/geocode_results_bad_zips.xlsx')

In [9]:
student_zip_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                58 non-null     int64  
 1   name              58 non-null     str    
 2   address           58 non-null     str    
 3   city              58 non-null     str    
 4   state             57 non-null     str    
 5   zip               57 non-null     float64
 6   longitude         52 non-null     float64
 7   latitude          52 non-null     float64
 8   complete_address  57 non-null     str    
 9   location          42 non-null     str    
 10  point             42 non-null     str    
dtypes: float64(3), int64(1), str(7)
memory usage: 5.1 KB


### Data Set 4

The fourth venue location data set was derived by Information Specialist Angelina Cortes using R with the Tidyverse library. This was shared as an Excel spreadsheet and incorporated into the revised master.

In [10]:
revised_master_sheet_names = pd.ExcelFile('input-data/product1_data_master_rev.xlsx').sheet_names
revised_master_sheet_names

['Ven_Ev_Ar_Address',
 'Venue_Artist_Event',
 'Layer 1 Venues',
 'artistsGenre',
 'genreIndex',
 'Sheet5',
 'artistsIndex',
 'Venues No Address',
 'venues_with_coordinates']

In [11]:
tidyverse_data = pd.read_excel('input-data/product1_data_master_rev.xlsx', sheet_name='venues_with_coordinates')
tidyverse_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Venue_Name    974 non-null    str    
 1   Longitude     964 non-null    object 
 2   Latitude      964 non-null    object 
 3   Address       974 non-null    str    
 4   Zip           967 non-null    float64
 5   City          970 non-null    str    
 6   State         973 non-null    str    
 7   ...8          0 non-null      float64
 8   full_address  974 non-null    str    
 9   needs_geo     974 non-null    bool   
 10  Unnamed: 10   2 non-null      str    
dtypes: bool(1), float64(2), object(2), str(6)
memory usage: 77.2+ KB


## Evaluating Data Reliability

Since the coordinate data was pulled from numerous different sources, and many of the data sets have different coordinates for the same venue, we need some way of evaluating the reliability of the sets to inform our choices for what data to keep for the final presentation version.

### Evaluating Mapbox Data

For example, I suspect that the Mapbox API, queried in March 2024, generated some false matches based on names of venues. I'll examine how far from San Antonio each of the coordinate sets is. This should be able to flag false positives that match venue name but are in fact not anywhyere near San Antonio.

In [12]:
SA_LAT = 29.42861139590851
SA_LON = -98.49619652095998

def haversine_miles(lat1, lon1, lat2, lon2):
    """Vectorized great-circle distance in miles. Returns NaN where inputs are NaN."""
    R = 3958.8  # Earth radius in miles
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

distance_from_sa_df = mapbox_data[['id', 'name', 'longitude', 'latitude']].copy()

distance_from_sa_df['distance_from_sa_miles'] = haversine_miles(
    distance_from_sa_df['latitude'], distance_from_sa_df['longitude'], SA_LAT, SA_LON
)

In [13]:
distance_from_sa_df

,id,name,longitude,latitude,distance_from_sa_miles
0,1,1033 Club,-98.480763,29.436116,1.063696
1,2,151 Saloon,-98.690386,29.464097,11.938526
2,3,1902 Nightclub,-98.478249,29.420607,1.213463
3,4,210 Kapone's,-98.478440,29.425190,1.094420
4,5,4th Quarter Sports Bar,-98.569389,29.524564,7.958401
...,...,...,...,...,...
969,970,Don Strange Ranch,-98.785587,29.889198,36.258256
970,971,San Fernando Cathedral,-98.492959,29.424739,0.330981
971,972,St. Anthony Hotel,-98.489474,29.427267,0.415083
972,973,The Warehouse Music Venue,-98.477108,29.425226,1.172316


Now that I have a new table with distances in miles from San Antonio, I can filter the data to show suspiciously distant venues.

In [14]:
too_far = distance_from_sa_df['distance_from_sa_miles'] > 35
distance_from_sa_df[too_far]

,id,name,longitude,latitude,distance_from_sa_miles
19,20,Alexander's Bar and Café,-62.184068,16.653923,2459.058117
79,80,Bo's Rodeo,-99.072459,29.727007,40.300707
107,108,Cabaret,-99.072459,29.727007,40.300707
322,323,Jack's Bar,98.439556,29.574315,8210.138480
346,347,Just Country,-94.642420,30.146324,236.338997
351,352,Kelly Field Terrace,-100.872166,32.400862,249.001031
356,357,Keystone Club,-98.871984,30.275201,62.679344
365,366,Krueger's Bar and Party House,-94.642420,30.146324,236.338997
372,373,La Rumba Nite Club,-96.562530,29.705360,117.767996
382,383,Lartos Lounge,-95.900044,33.247134,305.064051


Examining all the venues in the Mapbox data that are further than 35 miles from San Antonio yields some interesting results. 

Venues between 35 and 60 miles away appear to be correctly identifed, simply distant from San Antonio. For example, the Quiet Valley Ranch is indeed in Kerrville, as is Rocky Joe's. Schreiner University, likewise, is just that far. These are genuine concert venues that just happen to be somewhat outside San Antonio but are still in our concert history data set. 

On the other hand, venues further than 60 miles are mostly false positives. For example, Lyndy's Great Indoors is on Fredericksburg, not 62 miles away. Sumthin' Different is on Walzem, not 62 miles away. Venues identified as hundreds, if not thousands of miles away are certainly misidentifications.

The one exception is the Erwin Center, which is accurately placed 74 miles away.

In [15]:
len(distance_from_sa_df[too_far])

53

This analysis suggests the Mapbox data generated erroneous data for 52 venues (53 are further than 60 miles, but we exclude the Erwin Center from the list of bad data).

In [16]:
distance_from_sa_df['longitude'].isna().sum()

np.int64(71)

In addition, the Mapbox query failed to find coordinate data for 71 venues. 

We thus have a total of 124 venues out of 974 with bad coordinate data. That's roughly a 12.7% fail rate.

### Evaluating Tidyverse Data

I'll run the same analysis on the Tidyverse data.

When I try to run the analysis, I get an error that suggests one or more of the coordinates is not being stored as a float value. I'll check the data types in the Latitude and Longitude columns to see if I can identify the problem. This shows one longitude cell and one latitude cell are strings, not integers, and they are not necessarily in the same row.

I find out which are the offending rows - the Copper Penny for longitude and MacArthur field for latitude - and fix the issues. It looks like each of these entires got longitude-latitude pairs in one of the cells instead of individual numbers, and pandas has interpreted these as strings, which forces the entire column to be the vague "object" type.

In [17]:
# Inspect what's actually in there
print(tidyverse_data['Longitude'].apply(type).value_counts())
print(tidyverse_data['Latitude'].apply(type).value_counts())

Longitude
<class 'float'>    973
<class 'str'>        1
Name: count, dtype: int64
Latitude
<class 'float'>    973
<class 'str'>        1
Name: count, dtype: int64


In [18]:
non_float_data = tidyverse_data[tidyverse_data['Longitude'].apply(lambda x: isinstance(x, str))]
non_float_data

,Venue_Name,Longitude,Latitude,Address,Zip,City,State,...8,full_address,needs_geo,Unnamed: 10
705,The Copper Penny,"29.516951063329095, -98.45053963558209",29.516951,1217 NE IH410,78209.0,San Antonio,Texas,NaN,"1217 NE IH 410, NA, Texas, 78209",True,NaN


In [19]:
tidyverse_data.at[705, 'Longitude'] = -98.45053963558209
tidyverse_data.at[705, 'Latitude'] = 29.516951063329095
tidyverse_data.iloc[705]

Venue_Name                      The Copper Penny
Longitude                              -98.45054
Latitude                               29.516951
Address                            1217 NE IH410
Zip                                      78209.0
City                                 San Antonio
State                                      Texas
...8                                         NaN
full_address    1217 NE IH 410, NA, Texas, 78209
needs_geo                                   True
Unnamed: 10                                  NaN
Name: 705, dtype: object

Now we can run the distance analysis on the Tidyverse data.

In [20]:
bad_lat_data = tidyverse_data[tidyverse_data['Latitude'].apply(lambda x: isinstance(x, str))]
bad_lat_data

,Venue_Name,Longitude,Latitude,Address,Zip,City,State,...8,full_address,needs_geo,Unnamed: 10
935,MacArthur Field - Fort Sam Houston,-98.447957,"29.46083663942772, -98.44795724965459",3630 Stanley Rd,78234.0,Fort Sam Houston,Texas,NaN,"3630 Stanley Rd, Fort Sam Houston, Texas, 78234",True,NaN


In [21]:
tidyverse_data.at[935, 'Longitude'] = -98.44795724965459
tidyverse_data.at[935, 'Latitude'] = 29.46083663942772
tidyverse_data.iloc[935]

Venue_Name                   MacArthur Field - Fort Sam Houston
Longitude                                            -98.447957
Latitude                                              29.460837
Address                                         3630 Stanley Rd
Zip                                                     78234.0
City                                           Fort Sam Houston
State                                                     Texas
...8                                                        NaN
full_address    3630 Stanley Rd, Fort Sam Houston, Texas, 78234
needs_geo                                                  True
Unnamed: 10                                                 NaN
Name: 935, dtype: object

Now that these little data issues are resolved, I can coerce the latitude and longitude columns into numeric data types.

In [22]:
tidyverse_data['Longitude'] = pd.to_numeric(tidyverse_data['Longitude'])
tidyverse_data['Latitude'] = pd.to_numeric(tidyverse_data['Latitude'])
print(tidyverse_data['Longitude'].dtype)
print(tidyverse_data['Latitude'].dtype)

float64
float64


With all this data cleaning and data type conversion done, I create a copy of the data frame and add a column representing the distance from San Antonio. We can see here that the Tidyverse data appears more reliable and trustworthy than the Mapbox data. There is only one venue, Jack's Bar, that is not correctly geolocated. In addition, there are 10 venues that were not geolocated at all. That's 11 out of 974, which is a 1.1% fail rate compared to Mapbox's 12.7%. 

In [23]:
distance_from_sa_tidyverse = tidyverse_data[['Venue_Name', 'Longitude', 'Latitude']].copy()

In [24]:
distance_from_sa_tidyverse['distance_from_sa_miles'] = haversine_miles(
    distance_from_sa_tidyverse['Latitude'], distance_from_sa_tidyverse['Longitude'], SA_LAT, SA_LON
)

In [25]:
too_far_tidyverse = distance_from_sa_tidyverse['distance_from_sa_miles'] > 35
distance_from_sa_tidyverse[too_far_tidyverse]

,Venue_Name,Longitude,Latitude,distance_from_sa_miles
79,Bo's Rodeo,-99.075839,29.599295,36.793986
107,Cabaret,-99.067836,29.722193,39.892808
322,Jack's Bar,98.439556,29.574315,8210.138480
353,Kendalia Halle,-98.524228,29.968176,37.318638
449,Medina County Fairgrounds,-99.141930,29.350067,39.251417
856,Barney's-Bandera,-99.075000,29.725800,40.389694
863,Quiet Valley Ranch,-99.215735,29.958818,56.632023
907,Erwin Events Center,-97.733269,30.276865,74.331665
953,Rocky Joe's-Kerrville,-99.123740,30.062227,57.739338
954,Schreiner University,-99.129318,30.032002,56.401073


In [26]:
distance_from_sa_tidyverse['Longitude'].isna().sum()

np.int64(10)

Based on this analysis, Tidyverse results in better geolocation data than Mapbox, and in the future should be privileged as the main API to use in geolocating San Antonio venues.

### Evaluating Student Data

Now I'll switch to evaluating the location data derived by the student worker through research in newspaper archives.

#### Student Address Data

The first set of student data reflects work done to identify or improve the set of venues that were either missing addresses or had incomplete or unusual address information in the `Layer 1 Venues` spreadsheet. The list was based on a spreadsheet that contained lists of venues that were missing coordinate, address, or zip data.

First I'll take a look at the source file with the original list of venues given to the student to work on. I'll load that list and take a look at it.

In [27]:
pd.ExcelFile('input-data/venues_separated.xlsx').sheet_names

['All Venues',
 'Missing Coordinates',
 '>30 Miles Away',
 'Missing or Bad Address',
 'Missing Zip']

In [28]:
missing_or_bad_address = pd.read_excel('input-data/venues_separated.xlsx', sheet_name='Missing or Bad Address')

In [29]:
missing_or_bad_address

,id,name,address,city,state,zip,longitude,latitude
0,7,Abracadabra,Central Park Mall,San Antonio,Texas,78216.0,-98.495141,29.424600
1,36,Avalon Grill,509.5 East Commerce,San Antonio,Texas,78205.0,-98.495141,29.424600
2,53,Big Bob's Burgers,​447 W. Hildebrand Ave.,San Antonio,Texas,78212.0,-98.511348,29.466460
3,80,Bo's Rodeo,Hwy 173,Bandera,Texas,78003.0,-99.072459,29.727007
4,81,Botika Peruvian-Asian Restaurant,"The Pearl, 303 Pearl Pkwy #111",San Antonio,Texas,78215.0,-98.479960,29.442498
5,126,Casey's Saloon and Dance,Riverwalk,San Antonio,Texas,78205.0,-98.488467,29.424256
6,127,Casino Club,N Zarzamora St & Culebra Rd,San Antonio,Texas,78201.0,-98.524926,29.445398
7,218,Edge of Town,Culebra and Ingram,San Antonio,Texas,78251.0,-98.665781,29.457955
8,236,Engle's Place,Nacogdoches Rd,San Antonio,Texas,78217.0,-98.495141,29.424600
9,352,Kelly Field Terrace,corner Mission and Mitchell Road,NaN,Texas,NaN,-100.872166,32.400862


In [30]:
student_address_data

,id,name,address,city,state,zip,longitude,latitude,complete_address,location,point
0,7,Abracadabra,258 Central Park Mall,San Antonio,Texas,78216,-98.495141,29.424600,"258 Central Park Mall San Antonio, Texas 78216",NaN,NaN
1,36,Avalon Grill,509 1/2 E Commerce St,San Antonio,Texas,78205,-98.495141,29.424600,"509 1/2 E Commerce St San Antonio, Texas 78205","East Commerce Street, Downtown, San Antonio, B...","(29.4235382, -98.4869164)"
2,53,Big Bob's Burgers,2215 Harry Wurzbach Road,San Antonio,Texas,78212,-98.511348,29.466460,"2215 Harry Wurzbach Road San Antonio, Texas 78212","Harry Wurzbach Road, San Antonio, Bexar County...","(29.4624838, -98.4480441)"
3,80,Bo's Rodeo,Hwy 173,Bandera,Texas,78003,-99.072459,29.727007,"Hwy 173 Bandera, Texas 78003",NaN,NaN
4,81,Botika Peruvian-Asian Restaurant,"303 Pearl Parkway, Suite 111",San Antonio,Texas,78215,-98.479960,29.442498,"303 Pearl Parkway, Suite 111 San Antonio, Texa...",NaN,NaN
5,126,Casey's Saloon and Dance,West Crockett Street,San Antonio,Texas,78205,-98.488467,29.424256,"West Crockett Street San Antonio, Texas 78205","West Crockett Street, Downtown, San Antonio, B...","(29.4249236, -98.4905331)"
6,127,Casino Club,102 W Crockett St.,San Antonio,Texas,78205,-98.524926,29.445398,"102 W Crockett St. San Antonio, Texas 78205","102, West Crockett Street, Downtown, San Anton...","(29.4249639, -98.4911898)"
7,218,Edge of Town,Culebra and Ingram,San Antonio,Texas,78251,-98.665781,29.457955,"Culebra and Ingram San Antonio, Texas 78251",NaN,NaN
8,236,Engle's Place,Nacogdoches Rd,San Antonio,Texas,78217,-98.495141,29.424600,"Nacogdoches Rd San Antonio, Texas 78217","Texas Driving School, 2453, Nacogdoches Road, ...","(29.5180256, -98.451139)"
9,352,Kelly Field Terrace,corner Mission and Mitchell Road,San Antonio,Texas,78214,-100.872166,32.400862,"corner Mission and Mitchell Road San Antonio, ...",NaN,NaN


Now I'll compare the two sets of addresses to see how many have been fixed.

In [31]:
address_comparison = pd.merge(
    missing_or_bad_address[['id', 'name', 'address']].rename(columns={'address': 'orig_address'}),
    student_address_data[['id', 'address']].rename(columns={'address': 'student_address'}),
    on='id',
    how='left'
)

In [32]:
address_comparison

,id,name,orig_address,student_address
0,7,Abracadabra,Central Park Mall,258 Central Park Mall
1,36,Avalon Grill,509.5 East Commerce,509 1/2 E Commerce St
2,53,Big Bob's Burgers,​447 W. Hildebrand Ave.,2215 Harry Wurzbach Road
3,80,Bo's Rodeo,Hwy 173,Hwy 173
4,81,Botika Peruvian-Asian Restaurant,"The Pearl, 303 Pearl Pkwy #111","303 Pearl Parkway, Suite 111"
5,126,Casey's Saloon and Dance,Riverwalk,West Crockett Street
6,127,Casino Club,N Zarzamora St & Culebra Rd,102 W Crockett St.
7,218,Edge of Town,Culebra and Ingram,Culebra and Ingram
8,236,Engle's Place,Nacogdoches Rd,Nacogdoches Rd
9,352,Kelly Field Terrace,corner Mission and Mitchell Road,corner Mission and Mitchell Road


In comparing the addresses, there are 14 addresses that appear to be improvements over the old addresses, pending coordinate analysis.

In [33]:
improved_addresses = address_comparison[address_comparison['id'].isin([7, 36, 81, 127, 391, 461, 519, 533, 583, 615, 630, 649, 931, 956])]
improved_addresses

,id,name,orig_address,student_address
0,7,Abracadabra,Central Park Mall,258 Central Park Mall
1,36,Avalon Grill,509.5 East Commerce,509 1/2 E Commerce St
4,81,Botika Peruvian-Asian Restaurant,"The Pearl, 303 Pearl Pkwy #111","303 Pearl Parkway, Suite 111"
6,127,Casino Club,N Zarzamora St & Culebra Rd,102 W Crockett St.
10,391,Trinity University,One Trinity Pl,1 Trinity Pl
11,461,Moulin Rouge Lounge,1921.5 Fredricksburg Road,1921 1/2 Fredricksburg Road
13,519,Playpen North,San Pedro,1015 Jackson Keller Suite 116
15,533,R & J Lounge,Co Road 4517,18086 Pleasanton Rd
17,583,Ruth Taylor Theater,One Trintiy Place,715 Stadium Drive
18,615,Seven Oaks,Austin HWY and Military Road,1400 Austin HWY


Next, I'll examine the cooordinates given in that data set. 

In [34]:
coordinate_comparison = pd.merge(
    missing_or_bad_address[['id', 'name', 'longitude', 'latitude']].rename(columns={'longitude': 'orig_lon', 'latitude': 'orig_lat'}),
    student_address_data[['id', 'point']].rename(columns={'point': 'revised_coords'}),
    on='id',
    how='left'
)

coordinate_comparison

,id,name,orig_lon,orig_lat,revised_coords
0,7,Abracadabra,-98.495141,29.424600,NaN
1,36,Avalon Grill,-98.495141,29.424600,"(29.4235382, -98.4869164)"
2,53,Big Bob's Burgers,-98.511348,29.466460,"(29.4624838, -98.4480441)"
3,80,Bo's Rodeo,-99.072459,29.727007,NaN
4,81,Botika Peruvian-Asian Restaurant,-98.479960,29.442498,NaN
5,126,Casey's Saloon and Dance,-98.488467,29.424256,"(29.4249236, -98.4905331)"
6,127,Casino Club,-98.524926,29.445398,"(29.4249639, -98.4911898)"
7,218,Edge of Town,-98.665781,29.457955,NaN
8,236,Engle's Place,-98.495141,29.424600,"(29.5180256, -98.451139)"
9,352,Kelly Field Terrace,-100.872166,32.400862,NaN


It looks like there is one venue that did not have any coordinates that now does have coordinates, Sadler's Tent Theater. There also appear to be some improved locations, for example "Farmer's Market" is now accurate when it formerly was not. As a way of getting a sense for where improvements have been made, I'll run a distance analysis comparing the old coordinate data to the new coordinate data.

First, I'll create a copy of the data with only the rows that have new coordinate data.

In [35]:
revised_coord_data = coordinate_comparison[coordinate_comparison['revised_coords'].notnull()]
revised_coord_data

,id,name,orig_lon,orig_lat,revised_coords
1,36,Avalon Grill,-98.495141,29.424600,"(29.4235382, -98.4869164)"
2,53,Big Bob's Burgers,-98.511348,29.466460,"(29.4624838, -98.4480441)"
5,126,Casey's Saloon and Dance,-98.488467,29.424256,"(29.4249236, -98.4905331)"
6,127,Casino Club,-98.524926,29.445398,"(29.4249639, -98.4911898)"
8,236,Engle's Place,-98.495141,29.424600,"(29.5180256, -98.451139)"
10,391,Trinity University,-98.482574,29.463303,"(29.4652193, -98.4831256)"
12,478,Oak Grove Hall,-98.457242,29.524766,"(29.5180256, -98.451139)"
15,533,R & J Lounge,-99.030950,29.386590,"(29.244676, -98.498286)"
17,583,Ruth Taylor Theater,-98.483775,29.463766,"(29.4621639, -98.4812215)"
18,615,Seven Oaks,-97.743700,30.271129,"(29.491151, -98.439024)"


Now, I calculate the distance in miles form San Antonio for the original data. I add the new coordinate data to the table and do the same calculation for the new coordinates. I merge the sheets and place those values side by side for easy visual inspection. Then I add the same data from the tidyverse coordinate analysis, so all three data sources can be inspected at the same time. I rename some columns and drop other columns to make it simpler.

In [36]:
revised_coord_data['orig_dist'] = haversine_miles(
    revised_coord_data['orig_lat'], revised_coord_data['orig_lon'], SA_LAT, SA_LON
)

new_coords = revised_coord_data['revised_coords'].str.strip('()').str.split(', ', expand=True)
revised_coord_data['rev_lat'] = new_coords[0].astype(float)
revised_coord_data['rev_lon'] = new_coords[1].astype(float)

revised_coord_data['new_dist'] = haversine_miles(
    revised_coord_data['rev_lat'], revised_coord_data['rev_lon'], SA_LAT, SA_LON
)

revised_coord_analysis = revised_coord_data.merge(
    distance_from_sa_tidyverse[['Venue_Name', 'distance_from_sa_miles']],
    left_on='name',
    right_on='Venue_Name',
    how='left'
).drop(columns='Venue_Name')

revised_coord_analysis = revised_coord_analysis.rename(columns={'distance_from_sa_miles': 'tidyverse_dist'})   
revised_coord_analysis = revised_coord_analysis.drop(columns=['orig_lon', 'orig_lat', 'revised_coords', 'rev_lat', 'rev_lon'])
revised_coord_analysis

,id,name,orig_dist,new_dist,tidyverse_dist
0,36,Avalon Grill,0.284350,0.659371,0.615824
1,53,Big Bob's Burgers,2.769457,3.724456,2.769457
2,126,Casey's Saloon and Dance,0.554039,0.425543,0.594830
3,127,Casino Club,2.081790,0.392808,2.081790
4,236,Engle's Place,0.284350,6.746362,11.845062
5,391,Trinity University,2.533250,2.648833,2.533250
6,478,Oak Grove Hall,7.044799,6.746362,6.778572
7,533,R & J Lounge,32.318125,12.709473,32.268096
8,583,Ruth Taylor Theater,2.541358,2.487223,2.541358
9,615,Seven Oaks,73.635972,5.522889,NaN


This data set presents major challenges for deciding which coordinate set to consider authoritative. 

In the case of Big Bob's Burgers, I am not sure why my Python algorithm flagged it as having a bad address, since the original listed address was correct. The student revised the Hildebrand Avenue address to Harry Wurzbach Road, but this is not correct, as I used to live in the neighborhood with that location and eat burgers there while live music was playing.

For Zanzibar Nite Club, the Mapbox location is evidently wrong, and Tidyverse could not get the address, but the student revision, 1.8 miles from downtown, is suspect because the address data we have states that it was "Location, six blocks out of city limits, off Risgby, on Sulphur Spring road." If it's outside the city limits, it is necessarily more than 1.8 miles from the center of the city. Therefore, we still don't have valid coordinates for this venue.

For Sadler's Tent Theater, the student coordinates match the Tidyverse coordinates, which suggests accuracy.

For Seven Oaks, the student data appears to be the most likely, as Mapbox has it too far while Tidyverse did not retrieve a location.

Likewise for Sunken Garden, the student data is more valid than either API, as it is probably about 2 miles from downtown, whereas the APIs have it 17 miles (Mapbox) and 7 miles (Tidyverse) away.

In other cases, I have no obvious way to judge which coordinates are more valid. For example, Engle's Place or St. Hedwig Hall present different coordinates for all three data sets. 

I conclude that I will have to go through these locations one by one to determine the best choices, manually, as there is no self-evident "best" data source. In some cases the student appears to have determined the correct location, but in other cases clearly not.

#### Student Zip Code Data

I can now examine the student Zip Code data. My analysis of missing location information, in early summer 2025, found 58 venues missing zip codes in the `Layer 1 Venues` spreadsheet. The student stated in the PDF they shared that they used a USPS zip lookup tool, and filled in all but one missing Zips.

In addition, though, the data set as received has new coordinates, which in some cases differ significantly from the old coordinates. I'll repeat the analysis done for the address-based coordinates, in order to see if I can find any patterns that would allow for decisions to be made regarding which data set to favor.

In [37]:
student_zip_data

,id,name,address,city,state,zip,longitude,latitude,complete_address,location,point
0,320,Ivory Lounge & Nightclub,1201 Austin Highway,San Antonio,Texas,78209.0,-98.446810,29.489438,"1201 Austin Highway San Antonio, Texas 78209","1201, Austin Highway, San Antonio, Bexar Count...","(29.487423, -98.447385)"
1,347,Just Country,9323 Perrin Beitel,San Antonio,Texas,78217.0,-94.642420,30.146324,"9323 Perrin Beitel San Antonio, Texas 78217",Anchored SA - The San Antonio Community Church...,"(29.5251293, -98.4123022)"
2,352,Kelly Field Terrace,corner Mission and Mitchell Road,San Antonio,Texas,78214.0,-100.872166,32.400862,"corner Mission and Mitchell Road San Antonio, ...",NaN,NaN
3,357,Keystone Club,5152 Fredericksburg,San Antonio,Texas,78229.0,-98.871984,30.275201,"5152 Fredericksburg San Antonio, Texas 78229","5152, Fredericksburg Road, San Antonio, Bexar ...","(29.5021189, -98.5596)"
4,366,Krueger's Bar and Party House,6870 East Highway 87,San Antonio,Texas,78263.0,-94.642420,30.146324,"6870 East Highway 87 San Antonio, Texas 78263",NaN,NaN
5,373,La Rumba Nite Club,411 Old Hwy 90,San Antonio,Texas,78237.0,-96.562530,29.705360,"411 Old Hwy 90 San Antonio, Texas 78237","411, Historic Old Highway 90, San Antonio, Bex...","(29.4286292, -98.5687666)"
6,381,Larry's Cantina,10929 Nacogdoches,San Antonio,Texas,78217.0,NaN,NaN,"10929 Nacogdoches San Antonio, Texas 78217","10929, Nacogdoches Road, San Antonio, Bexar Co...","(29.5388177, -98.4232872)"
7,382,Larry's Lounge,2219 South Flores,San Antonio,Texas,78204.0,NaN,NaN,"2219 South Flores San Antonio, Texas 78204","2219, South Flores Street, Blue Star, San Anto...","(29.402161, -98.503771)"
8,383,Lartos Lounge,2303 East Commerce,San Antonio,Texas,78203.0,-95.900044,33.247134,"2303 East Commerce San Antonio, Texas 78203","2303, East Commerce Street, San Antonio, Bexar...","(29.4195251, -98.458818)"
9,388,Last National Bank,10127 Coachlight,San Antonio,Texas,78216.0,NaN,NaN,"10127 Coachlight San Antonio, Texas 78216","10127, Coachlight Street, San Antonio, Bexar C...","(29.5340398, -98.4945622)"


The student's zip code data was aimed at resolving th

In [38]:
orig_missing_zips = pd.read_excel('input-data/venues_separated.xlsx', sheet_name='Missing Zip')
orig_missing_zips

,id,name,address,city,state,zip,longitude,latitude
0,320,Ivory Lounge & Nightclub,1201 Austin Highway,NaN,Texas,NaN,-98.446810,29.489438
1,347,Just Country,9323 Perrin Beitel,NaN,Texas,NaN,-94.642420,30.146324
2,352,Kelly Field Terrace,corner Mission and Mitchell Road,NaN,Texas,NaN,-100.872166,32.400862
3,357,Keystone Club,5152 Fredericksburg,NaN,Texas,NaN,-98.871984,30.275201
4,366,Krueger's Bar and Party House,6870 East Highway 87,NaN,Texas,NaN,-94.642420,30.146324
5,373,La Rumba Nite Club,411 Old Hwy 90,NaN,Texas,NaN,-96.562530,29.705360
6,381,Larry's Cantina,10929 Nacogdoches,NaN,Texas,NaN,NaN,NaN
7,382,Larry's Lounge,2219 South Flores,NaN,Texas,NaN,NaN,NaN
8,383,Lartos Lounge,2303 East Commerce,NaN,Texas,NaN,-95.900044,33.247134
9,388,Last National Bank,10127 Coachlight,NaN,Texas,NaN,NaN,NaN


It appears that in the original `Layer 1 Venues` sheet, there were a number of venues (58) missing zip codes, and most of them were also missing city data. Although most of them are (or were) in San Antonio, this is not always the case. The student filled in a lof of missing city data and added a "complete address" field in addition to zip codes.

We'll want to bring in all the missing city and zip data the student added.

Now I'll do a coordinate analysis. As was the case for the coordinates in the address sheet, the new coordinates here are stored as strings between parentheses, so the first task is to break those out and store them as floating point values. Working just with the non-null coordinate values to avoid problems, I strip off the parentheses, split them into a list format, and add the values to the table.

In [39]:
new_coords_basis = student_zip_data[student_zip_data['point'].notnull()]
new_coords = new_coords_basis['point'].str.strip('()').str.split(', ', expand=True)
student_zip_data['rev_lat'] = new_coords[0].astype(float)
student_zip_data['rev_lon'] = new_coords[1].astype(float)
student_zip_data

,id,name,address,city,state,zip,longitude,latitude,complete_address,location,point,rev_lat,rev_lon
0,320,Ivory Lounge & Nightclub,1201 Austin Highway,San Antonio,Texas,78209.0,-98.446810,29.489438,"1201 Austin Highway San Antonio, Texas 78209","1201, Austin Highway, San Antonio, Bexar Count...","(29.487423, -98.447385)",29.487423,-98.447385
1,347,Just Country,9323 Perrin Beitel,San Antonio,Texas,78217.0,-94.642420,30.146324,"9323 Perrin Beitel San Antonio, Texas 78217",Anchored SA - The San Antonio Community Church...,"(29.5251293, -98.4123022)",29.525129,-98.412302
2,352,Kelly Field Terrace,corner Mission and Mitchell Road,San Antonio,Texas,78214.0,-100.872166,32.400862,"corner Mission and Mitchell Road San Antonio, ...",NaN,NaN,NaN,NaN
3,357,Keystone Club,5152 Fredericksburg,San Antonio,Texas,78229.0,-98.871984,30.275201,"5152 Fredericksburg San Antonio, Texas 78229","5152, Fredericksburg Road, San Antonio, Bexar ...","(29.5021189, -98.5596)",29.502119,-98.559600
4,366,Krueger's Bar and Party House,6870 East Highway 87,San Antonio,Texas,78263.0,-94.642420,30.146324,"6870 East Highway 87 San Antonio, Texas 78263",NaN,NaN,NaN,NaN
5,373,La Rumba Nite Club,411 Old Hwy 90,San Antonio,Texas,78237.0,-96.562530,29.705360,"411 Old Hwy 90 San Antonio, Texas 78237","411, Historic Old Highway 90, San Antonio, Bex...","(29.4286292, -98.5687666)",29.428629,-98.568767
6,381,Larry's Cantina,10929 Nacogdoches,San Antonio,Texas,78217.0,NaN,NaN,"10929 Nacogdoches San Antonio, Texas 78217","10929, Nacogdoches Road, San Antonio, Bexar Co...","(29.5388177, -98.4232872)",29.538818,-98.423287
7,382,Larry's Lounge,2219 South Flores,San Antonio,Texas,78204.0,NaN,NaN,"2219 South Flores San Antonio, Texas 78204","2219, South Flores Street, Blue Star, San Anto...","(29.402161, -98.503771)",29.402161,-98.503771
8,383,Lartos Lounge,2303 East Commerce,San Antonio,Texas,78203.0,-95.900044,33.247134,"2303 East Commerce San Antonio, Texas 78203","2303, East Commerce Street, San Antonio, Bexar...","(29.4195251, -98.458818)",29.419525,-98.458818
9,388,Last National Bank,10127 Coachlight,San Antonio,Texas,78216.0,NaN,NaN,"10127 Coachlight San Antonio, Texas 78216","10127, Coachlight Street, San Antonio, Bexar C...","(29.5340398, -98.4945622)",29.534040,-98.494562


Now, I'll make a table with just the original coordinates and the new coordinates.

In [40]:
zip_data_coord_comparison = student_zip_data[student_zip_data['point'].notnull()]
zip_data_coord_comparison = pd.merge(
    orig_missing_zips[['id', 'name', 'longitude', 'latitude']].rename(columns={'longitude': 'orig_lon', 'latitude': 'orig_lat'}),
    student_zip_data[['id', 'rev_lat', 'rev_lon']],
    on='id',
    how='left'
)
zip_data_coord_comparison

,id,name,orig_lon,orig_lat,rev_lat,rev_lon
0,320,Ivory Lounge & Nightclub,-98.446810,29.489438,29.487423,-98.447385
1,347,Just Country,-94.642420,30.146324,29.525129,-98.412302
2,352,Kelly Field Terrace,-100.872166,32.400862,NaN,NaN
3,357,Keystone Club,-98.871984,30.275201,29.502119,-98.559600
4,366,Krueger's Bar and Party House,-94.642420,30.146324,NaN,NaN
5,373,La Rumba Nite Club,-96.562530,29.705360,29.428629,-98.568767
6,381,Larry's Cantina,NaN,NaN,29.538818,-98.423287
7,382,Larry's Lounge,NaN,NaN,29.402161,-98.503771
8,383,Lartos Lounge,-95.900044,33.247134,29.419525,-98.458818
9,388,Last National Bank,NaN,NaN,29.534040,-98.494562


Now, I'll calculate the distances.

In [41]:
zip_data_coord_comparison['orig_dist'] = haversine_miles(
    zip_data_coord_comparison['orig_lat'], zip_data_coord_comparison['orig_lon'], SA_LAT, SA_LON
)

zip_data_coord_comparison['new_dist'] = haversine_miles(
    zip_data_coord_comparison['rev_lat'], zip_data_coord_comparison['rev_lon'], SA_LAT, SA_LON
)

zip_data_coord_comparison = zip_data_coord_comparison.merge(
    distance_from_sa_tidyverse[['Venue_Name', 'distance_from_sa_miles']],
    left_on='name',
    right_on='Venue_Name',
    how='left'
).drop(columns='Venue_Name')

zip_data_coord_comparison = zip_data_coord_comparison.rename(columns={'distance_from_sa_miles': 'tidyverse_dist'})   
zip_data_coord_comparison = zip_data_coord_comparison.drop(columns=['orig_lon', 'orig_lat', 'rev_lat', 'rev_lon']) 
zip_data_coord_comparison

,id,name,orig_dist,new_dist,tidyverse_dist
0,320,Ivory Lounge & Nightclub,5.146921,5.013555,5.228117
1,347,Just Country,236.338997,8.362884,8.353263
2,352,Kelly Field Terrace,249.001031,NaN,2.482274
3,357,Keystone Club,62.679344,6.351646,6.374291
4,366,Krueger's Bar and Party House,236.338997,NaN,8.895714
5,373,La Rumba Nite Club,117.767996,4.367179,4.382074
6,381,Larry's Cantina,NaN,8.787052,8.861255
7,382,Larry's Lounge,NaN,1.883568,1.843424
8,383,Lartos Lounge,305.064051,2.335459,2.325168
9,388,Last National Bank,NaN,7.285143,7.263527


On visual inspection, these coordinates appear to be significantly more reliable than the ones linked to the revised address sheet. None of them are outside the boundaries of where the data should be. Where there are also Tidyverse coordinates, the two are close to one another. There are three cases where Tidyverse and/or Mapbox failed, but this manual process did not: Torres Dance Hall, Playpen North, and Mary's Cafe. In contrast, there are some cases where only one of the APIs provided data: specifically, 

I conclude that the coordinates and address data from this sheet are more reliable overall than either API source. 

The order of reliability as far as I can tell is
1. Student Zip Sheet Coordinates
2. Tidyverse Coordinates
3. Mapbox Coordinates (verify distance before using)
3. Student Address Sheet Coordinates (verify before using)

## Generate Complete List of Venue Locations

It's time now to generate a full list of all the venues that we have any location data for. I will begin with a simple two-column data frame consisting of the id and name of each venue.

In [42]:
venues_with_locations = pd.DataFrame({
    'id': mapbox_data['id'],
    'name': mapbox_data['name']
})

In [43]:
venues_with_locations

,id,name
0,1,1033 Club
1,2,151 Saloon
2,3,1902 Nightclub
3,4,210 Kapone's
4,5,4th Quarter Sports Bar
...,...,...
969,970,Don Strange Ranch
970,971,San Fernando Cathedral
971,972,St. Anthony Hotel
972,973,The Warehouse Music Venue


### Student Zip Data

Now I'll add information from the student zip data sheet. I'll inspect that table to make sure I don't use any null data. All the rows have address and city information. There is one row with null state and zip information. There are 42 sets of revised coordinate data. 

Based on this, it's safe to merge all the address and city data at once. The zip and coordinate data will have to go through an intermediate step where I eliminate rows with null values before merging.

In [44]:
student_zip_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                58 non-null     int64  
 1   name              58 non-null     str    
 2   address           58 non-null     str    
 3   city              58 non-null     str    
 4   state             57 non-null     str    
 5   zip               57 non-null     float64
 6   longitude         52 non-null     float64
 7   latitude          52 non-null     float64
 8   complete_address  57 non-null     str    
 9   location          42 non-null     str    
 10  point             42 non-null     str    
 11  rev_lat           42 non-null     float64
 12  rev_lon           42 non-null     float64
dtypes: float64(5), int64(1), str(7)
memory usage: 6.0 KB


In [45]:
venues_with_locations = venues_with_locations.merge(
    student_zip_data[['id', 'address', 'city']],
    on='id',
    how='left'
)

In [46]:
student_zip_non_null_state = student_zip_data[student_zip_data['state'].notnull()]
venues_with_locations = venues_with_locations.merge(
    student_zip_non_null_state[['id', 'state']],
    on='id',
    how='left'
)

In [47]:
student_zip_non_null_zip = student_zip_data[student_zip_data['zip'].notnull()]
venues_with_locations = venues_with_locations.merge(
    student_zip_non_null_zip[['id', 'zip']],
    on='id',
    how='left'
)

In [48]:
student_zip_non_null_coords = student_zip_data[student_zip_data['rev_lat'].notnull() & student_zip_data['rev_lon'].notnull()]
venues_with_locations = venues_with_locations.merge(
    student_zip_non_null_coords[['id', 'rev_lon', 'rev_lat']],
    on='id',
    how='left'
)

venues_with_locations = venues_with_locations.rename(columns={'rev_lon': 'longitude', 'rev_lat': 'latitude'})

In [49]:
venues_with_locations[venues_with_locations['longitude'].notnull()]

,id,name,address,city,state,zip,longitude,latitude
319,320,Ivory Lounge & Nightclub,1201 Austin Highway,San Antonio,Texas,78209.0,-98.447385,29.487423
346,347,Just Country,9323 Perrin Beitel,San Antonio,Texas,78217.0,-98.412302,29.525129
356,357,Keystone Club,5152 Fredericksburg,San Antonio,Texas,78229.0,-98.559600,29.502119
372,373,La Rumba Nite Club,411 Old Hwy 90,San Antonio,Texas,78237.0,-98.568767,29.428629
380,381,Larry's Cantina,10929 Nacogdoches,San Antonio,Texas,78217.0,-98.423287,29.538818
381,382,Larry's Lounge,2219 South Flores,San Antonio,Texas,78204.0,-98.503771,29.402161
382,383,Lartos Lounge,2303 East Commerce,San Antonio,Texas,78203.0,-98.458818,29.419525
387,388,Last National Bank,10127 Coachlight,San Antonio,Texas,78216.0,-98.494562,29.534040
395,396,Let's Go Bottom Up,542 Fredericksburg,San Antonio,Texas,78201.0,-98.509488,29.447635
400,401,Limon's Lounge,722 S. St. Mary's,San Antonio,Texas,78205.0,-98.488339,29.412121


### Tidyverse Data

Next I will add in data retrieved from Tidyverse.

In [50]:
tidyverse_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Venue_Name    974 non-null    str    
 1   Longitude     964 non-null    float64
 2   Latitude      964 non-null    float64
 3   Address       974 non-null    str    
 4   Zip           967 non-null    float64
 5   City          970 non-null    str    
 6   State         973 non-null    str    
 7   ...8          0 non-null      float64
 8   full_address  974 non-null    str    
 9   needs_geo     974 non-null    bool   
 10  Unnamed: 10   2 non-null      str    
dtypes: bool(1), float64(4), str(6)
memory usage: 77.2 KB


This will be a lot easier if the tidyverse data has an `id` column to match the one we are working with, so I'll add that before proceeding.

In [51]:
tidyverse_data['id'] = range(1, len(tidyverse_data) + 1)

In [52]:
tidyverse_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Venue_Name    974 non-null    str    
 1   Longitude     964 non-null    float64
 2   Latitude      964 non-null    float64
 3   Address       974 non-null    str    
 4   Zip           967 non-null    float64
 5   City          970 non-null    str    
 6   State         973 non-null    str    
 7   ...8          0 non-null      float64
 8   full_address  974 non-null    str    
 9   needs_geo     974 non-null    bool   
 10  Unnamed: 10   2 non-null      str    
 11  id            974 non-null    int64  
dtypes: bool(1), float64(4), int64(1), str(6)
memory usage: 84.8 KB


In [53]:
venues_with_locations.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         974 non-null    int64  
 1   name       974 non-null    str    
 2   address    58 non-null     str    
 3   city       58 non-null     str    
 4   state      57 non-null     str    
 5   zip        57 non-null     float64
 6   longitude  42 non-null     float64
 7   latitude   42 non-null     float64
dtypes: float64(3), int64(1), str(4)
memory usage: 61.0 KB


Next, I'll bring in the longitude and latitude data from the Tidyverse query. I don't want to overwrite any existing data. So, I'll merge the data, but into new temporary columns. Then I'll use those columns as sources of data to fill in null rows in the latitude and longitude columns, using the `.fillna()` method. Then I'll drop the temporary columns.

In [54]:
venues_with_locations = pd.merge(
    venues_with_locations, 
    tidyverse_data[['id', 'Longitude', 'Latitude']].rename(
        columns={'Longitude': 'tidy_lon', 'Latitude': 'tidy_lat'}
    ),
    on='id',
    how='left'
)

venues_with_locations['longitude'] = venues_with_locations['longitude'].fillna(venues_with_locations['tidy_lon'])
venues_with_locations['latitude'] = venues_with_locations['latitude'].fillna(venues_with_locations['tidy_lat'])
venues_with_locations = venues_with_locations.drop(columns=['tidy_lon', 'tidy_lat'])

In [55]:
venues_with_locations.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         974 non-null    int64  
 1   name       974 non-null    str    
 2   address    58 non-null     str    
 3   city       58 non-null     str    
 4   state      57 non-null     str    
 5   zip        57 non-null     float64
 6   longitude  968 non-null    float64
 7   latitude   968 non-null    float64
dtypes: float64(3), int64(1), str(4)
memory usage: 61.0 KB


### Mapbox Data

Next, I'll fill in data from the Mapbox query. I'll use this same source of data to fill in missing address information, since the address information in that data frame is the original one from `Layer 1 Venues`. 

I'll use the same strategy as for Tidyverse.

In [56]:
mapbox_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         974 non-null    int64  
 1   name       974 non-null    str    
 2   address    974 non-null    str    
 3   city       912 non-null    str    
 4   state      972 non-null    str    
 5   zip        916 non-null    float64
 6   longitude  903 non-null    float64
 7   latitude   903 non-null    float64
dtypes: float64(3), int64(1), str(4)
memory usage: 61.0 KB


In [57]:
venues_with_locations = pd.merge(
    venues_with_locations, 
    mapbox_data[['id', 'longitude', 'latitude', 'zip']].rename(
        columns={'longitude': 'map_lon', 'latitude': 'map_lat', 'zip': 'map_zip'}
    ),
    on='id',
    how='left'
)

venues_with_locations['longitude'] = venues_with_locations['longitude'].fillna(venues_with_locations['map_lon'])
venues_with_locations['latitude'] = venues_with_locations['latitude'].fillna(venues_with_locations['map_lat'])
venues_with_locations['zip'] = venues_with_locations['zip'].fillna(venues_with_locations['map_zip'])
venues_with_locations = venues_with_locations.drop(columns=['map_lon', 'map_lat', 'map_zip'])

In [58]:
venues_with_locations.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         974 non-null    int64  
 1   name       974 non-null    str    
 2   address    58 non-null     str    
 3   city       58 non-null     str    
 4   state      57 non-null     str    
 5   zip        973 non-null    float64
 6   longitude  974 non-null    float64
 7   latitude   974 non-null    float64
dtypes: float64(3), int64(1), str(4)
memory usage: 61.0 KB


In [59]:
venues_with_locations = pd.merge(
    venues_with_locations,
    mapbox_data[['id', 'address', 'city', 'state']].rename(
        columns={'address': 'map_address', 'city': 'map_city', 'state': 'map_state'}
    ),
    on='id',
    how='left'
)

venues_with_locations['address'] = venues_with_locations['address'].fillna(venues_with_locations['map_address'])
venues_with_locations['city'] = venues_with_locations['city'].fillna(venues_with_locations['map_city'])
venues_with_locations['state'] = venues_with_locations['state'].fillna(venues_with_locations['map_state'])

venues_with_locations = venues_with_locations.drop(columns=['map_address', 'map_city', 'map_state'])

In [60]:
venues_with_locations.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         974 non-null    int64  
 1   name       974 non-null    str    
 2   address    974 non-null    str    
 3   city       969 non-null    str    
 4   state      972 non-null    str    
 5   zip        973 non-null    float64
 6   longitude  974 non-null    float64
 7   latitude   974 non-null    float64
dtypes: float64(3), int64(1), str(4)
memory usage: 61.0 KB


### Student Address Data

Finally, I'll add in data from the student address table. As a reminder, these were 36 rows of data. I identified 14 rows that contained improved addresses. We'll start by putting those addresses in. 

In [61]:
student_address_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                36 non-null     int64  
 1   name              36 non-null     str    
 2   address           36 non-null     str    
 3   city              36 non-null     str    
 4   state             36 non-null     str    
 5   zip               36 non-null     int64  
 6   longitude         33 non-null     float64
 7   latitude          33 non-null     float64
 8   complete_address  36 non-null     str    
 9   location          21 non-null     str    
 10  point             21 non-null     str    
dtypes: float64(2), int64(2), str(7)
memory usage: 3.2 KB


Since we are considering these addresses improvements over existing addresses, we are going to overwrite existing data. To that end, instead of `merge` and `fillna` I'll use `.update()`. 

In [62]:
improved_address_ids = [7, 36, 81, 127, 391, 461, 519, 533, 583, 615, 630, 649, 931, 956]
improved_address_data = student_address_data[
    student_address_data['id'].isin(improved_address_ids)
    ][['id', 'address', 'city', 'state', 'zip']]

venues_with_locations = venues_with_locations.set_index('id')
improved_address_data = improved_address_data.set_index('id')

improved_address_data.index[improved_address_data.index.duplicated(keep=False)]

venues_with_locations.update(improved_address_data, overwrite=True)
venues_with_locations.reset_index()

,id,name,address,city,state,zip,longitude,latitude
0,1,1033 Club,1033 Avenue B,San Antonio,Texas,78215.0,-98.480763,29.436116
1,2,151 Saloon,10619 Westover Hills Blvd,San Antonio,Texas,78215.0,-98.690386,29.464097
2,3,1902 Nightclub,1174 East Commerce St,San Antonio,Texas,78205.0,-98.478216,29.421548
3,4,210 Kapone's,1223 East Houston,San Antonio,Texas,78205.0,-98.478815,29.425150
4,5,4th Quarter Sports Bar,8779 Wurzbach Rd,San Antonio,Texas,78240.0,-98.569389,29.524564
...,...,...,...,...,...,...,...,...
969,970,Don Strange Ranch,103 Waring Welfare Rd,Boerne,Texas,78006.0,-98.786455,29.888503
970,971,San Fernando Cathedral,115 Main Plaza,San Antonio,Texas,78205.0,-98.493194,29.424797
971,972,St. Anthony Hotel,300 E Travis St,San Antonio,Texas,78205.0,-98.489684,29.427490
972,973,The Warehouse Music Venue,1305 E. Houston St.,San Antonio,Texas,78205.0,-98.477629,29.425113


In [63]:
venues_with_locations.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 1 to 974
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       974 non-null    str    
 1   address    974 non-null    str    
 2   city       969 non-null    str    
 3   state      972 non-null    str    
 4   zip        973 non-null    float64
 5   longitude  974 non-null    float64
 6   latitude   974 non-null    float64
dtypes: float64(3), str(4)
memory usage: 53.4 KB


After this update, we have not added any new rows, only altered existing rows. We are left with 5 null cities, 2 null states, and 1 null zip. 

## Filling in the Blanks

Somehow, in the process the id column became the index and remained the index, so we don't have the id column any more. First, I'll fix that.

In [64]:
venues_with_locations = venues_with_locations.reset_index()
venues_with_locations.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         974 non-null    int64  
 1   name       974 non-null    str    
 2   address    974 non-null    str    
 3   city       969 non-null    str    
 4   state      972 non-null    str    
 5   zip        973 non-null    float64
 6   longitude  974 non-null    float64
 7   latitude   974 non-null    float64
dtypes: float64(3), int64(1), str(4)
memory usage: 61.0 KB


I run this script that searches all our data sources for the missing city data and fills it in if it finds it. It finds 4 out of the 5 missing cities in the Tidyverse data and fills them in.

In [65]:
data_sources = [
    ('student_zip_data', student_zip_data),
    ('tidyverse_data', tidyverse_data),
    ('mapbox_data', mapbox_data),
    ('student_address_data', student_address_data),
]

for label, source_df in data_sources:
    city = 'City' if 'City' in source_df.columns else 'city'

    city_lookup = source_df[['id', city]].rename(columns={city: 'city_fill'})
    city_lookup = city_lookup.dropna(subset=['city_fill'])
    city_lookup = city_lookup.drop_duplicates(subset='id', keep='first')

    venues_with_locations = pd.merge(venues_with_locations, city_lookup, on='id', how='left')
    before = venues_with_locations['city'].isna().sum()
    venues_with_locations['city'] = venues_with_locations['city'].fillna(venues_with_locations['city_fill'])
    after = venues_with_locations['city'].isna().sum()
    venues_with_locations = venues_with_locations.drop(columns='city_fill')

    print(f"{label}: filled {before - after} nulls, {after} remaining")

student_zip_data: filled 0 nulls, 5 remaining
tidyverse_data: filled 4 nulls, 1 remaining
mapbox_data: filled 0 nulls, 1 remaining
student_address_data: filled 0 nulls, 1 remaining


In [66]:
venues_with_locations.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         974 non-null    int64  
 1   name       974 non-null    str    
 2   address    974 non-null    str    
 3   city       973 non-null    str    
 4   state      972 non-null    str    
 5   zip        973 non-null    float64
 6   longitude  974 non-null    float64
 7   latitude   974 non-null    float64
dtypes: float64(3), int64(1), str(4)
memory usage: 61.0 KB


I'll do the same for missing zips.

In [67]:
for label, source_df in data_sources:
    zip = 'Zip' if 'Zip' in source_df.columns else 'zip'

    zip_lookup = source_df[['id', zip]].rename(columns={zip: 'zip_fill'})
    zip_lookup = zip_lookup.dropna(subset=['zip_fill'])
    zip_lookup = zip_lookup.drop_duplicates(subset='id', keep='first')

    venues_with_locations = pd.merge(venues_with_locations, zip_lookup, on='id', how='left')
    before = venues_with_locations['zip'].isna().sum()
    venues_with_locations['zip'] = venues_with_locations['zip'].fillna(venues_with_locations['zip_fill'])
    after = venues_with_locations['zip'].isna().sum()
    venues_with_locations = venues_with_locations.drop(columns='zip_fill')

    print(f"{label}: filled {before - after} nulls, {after} remaining")

student_zip_data: filled 0 nulls, 1 remaining
tidyverse_data: filled 1 nulls, 0 remaining
mapbox_data: filled 0 nulls, 0 remaining
student_address_data: filled 0 nulls, 0 remaining


The script found the last missing zip code in the Tidyverse data and filled it in.

For states, we can make the assumption that all these concerts are in Texas, but I'll still run the script.

In [68]:
for label, source_df in data_sources:
    state = 'State' if 'State' in source_df.columns else 'state'

    state_lookup = source_df[['id', state]].rename(columns={state: 'state_fill'})
    state_lookup = state_lookup.dropna(subset=['state_fill'])
    state_lookup = state_lookup.drop_duplicates(subset='id', keep='first')

    venues_with_locations = pd.merge(venues_with_locations, state_lookup, on='id', how='left')
    before = venues_with_locations['state'].isna().sum()
    venues_with_locations['state'] = venues_with_locations['state'].fillna(venues_with_locations['state_fill'])
    after = venues_with_locations['state'].isna().sum()
    venues_with_locations = venues_with_locations.drop(columns='state_fill')

    print(f"{label}: filled {before - after} nulls, {after} remaining")

student_zip_data: filled 0 nulls, 2 remaining
tidyverse_data: filled 1 nulls, 1 remaining
mapbox_data: filled 0 nulls, 1 remaining
student_address_data: filled 0 nulls, 1 remaining


After running the script, we are left with one empty state value. I'll search for it.

In [69]:
venues_with_locations.loc[venues_with_locations['state'].isna()]

,id,name,address,city,state,zip,longitude,latitude
945,946,Tejano Explosion 2015,1000 W. Commerce St,San Antonio,NaN,78205.0,-98.505276,29.426412


It's in San Antonio, and we can be 100% certain it's Texas, so we'll manually add that.

In [70]:
venues_with_locations.loc[945, 'state'] = 'Texas'

In [71]:
venues_with_locations.loc[venues_with_locations['id'] == 946]

,id,name,address,city,state,zip,longitude,latitude
945,946,Tejano Explosion 2015,1000 W. Commerce St,San Antonio,Texas,78205.0,-98.505276,29.426412


In [72]:
venues_with_locations.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         974 non-null    int64  
 1   name       974 non-null    str    
 2   address    974 non-null    str    
 3   city       973 non-null    str    
 4   state      974 non-null    str    
 5   zip        974 non-null    float64
 6   longitude  974 non-null    float64
 7   latitude   974 non-null    float64
dtypes: float64(3), int64(1), str(4)
memory usage: 61.0 KB


We're left with just one null city. I'll take a look at it.

In [73]:
venues_with_locations.loc[venues_with_locations['city'].isna()]

,id,name,address,city,state,zip,longitude,latitude
690,691,Texco Nite Club,625 N Medina,NaN,Texas,78207.0,-98.504515,29.431389


This is definitely in San Antonio, so I'll fill that in manually.

In [74]:
venues_with_locations.loc[690, 'city'] = 'San Antonio'

In [75]:
venues_with_locations.loc[venues_with_locations['id'] == 691]

,id,name,address,city,state,zip,longitude,latitude
690,691,Texco Nite Club,625 N Medina,San Antonio,Texas,78207.0,-98.504515,29.431389


There's just one housekeeping task left. The Zip column contains floating point values, we'll want to convert that to integers.

In [76]:
venues_with_locations['zip'] = venues_with_locations['zip'].astype(int)

In [77]:
venues_with_locations.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         974 non-null    int64  
 1   name       974 non-null    str    
 2   address    974 non-null    str    
 3   city       974 non-null    str    
 4   state      974 non-null    str    
 5   zip        974 non-null    int64  
 6   longitude  974 non-null    float64
 7   latitude   974 non-null    float64
dtypes: float64(2), int64(2), str(4)
memory usage: 61.0 KB


There's a lingering issue. Recall that when I ran the distance analysis, Zanzibar Nite Club showed up as 1.8 miles from central San Antonio in the only data set to have a location for it, the student address data. But this was not possible based on the available address data, which placed it "six blocks out of city limits, off Risgby, on Sulphur Spring road." This is a problem, so I do some hands-on investigation.

In the data, the Zanzibar appears as the location of a few concerts in the 1940s. So I search for maps of San Antonio from that time or a bit later. If we look at Rigsby Avenue today, it turns into Highway 87 as it leaves San Antonio towards China Grove out east. Sulphur Springs Road currently runs parallel to Rigsby, south of it, and never intersects it.

Searching through our UTSA [LibGuide to San Antonio Maps](https://libguides.utsa.edu/c.php?g=512003&p=8635695), I find the maps in the [UNT Portal to Texas History](https://texashistory.unt.edu/search/?q=%22san+antonio%22&fq=dc_type%3Aimage%2A&fq=dc_type%3Aimage_map&t=metadata). Narrowing it down by decade, I find a [map](https://texashistory.unt.edu/ark:/67531/metapth450552/m1/1/zoom/?q=%22san%20antonio%22&resolution=2.0208503652985415&lat=2637.863098444719&lon=4854.497515663562) that shows the area in question in the 1950s. At that time, Rigsby did intersect Sulphur Springs Road. "Six blocks out of city limits" would have been two blocks past Rigsby. 

The identifying features that still exist today are Salado Creek and Eastview Cemetery. The Stewart School is also still there. This means that what was then Sulphur Springs Road is now Roland Avenue. Two blocks past Rigsby, about six blocks from where the city limits were, would be about where Amity Road intersects Roland Avenue today. That's 29.39414799158722, -98.42951980146205. The current Zip Code is 78210. I'll add that to the data.

In [78]:
venues_with_locations.loc[venues_with_locations['id'] == 836]

,id,name,address,city,state,zip,longitude,latitude
835,836,Zanzibar Nite Club,"six blocks out of city limits, off Risgby, on ...",San Antonio,Texas,78263,-95.417255,30.079883


In [79]:
venues_with_locations.loc[venues_with_locations['id'] == 836, ['zip', 'longitude', 'latitude']] = [78210, -98.4295198, 29.39414799]

In [80]:
venues_with_locations.loc[venues_with_locations['id'] == 836]

,id,name,address,city,state,zip,longitude,latitude
835,836,Zanzibar Nite Club,"six blocks out of city limits, off Risgby, on ...",San Antonio,Texas,78210,-98.42952,29.394148


## Quality Control

Before writing to the Excel file, I want to run a final distance test to verify that no really bad data has snuck into the data frame.

In [86]:
distance_test_df = pd.DataFrame({
    'id': venues_with_locations['id'],
    'name': venues_with_locations['name'],
    'lon': venues_with_locations['longitude'],
    'lat': venues_with_locations['latitude']
})

distance_test_df['dist'] = haversine_miles(
    distance_test_df['lat'], distance_test_df['lon'], SA_LAT, SA_LON
)

far = distance_test_df['dist'] > 60
distance_test_df[far]

,id,name,lon,lat,dist
322,323,Jack's Bar,98.439556,29.574315,8210.138480
614,615,Seven Oaks,-97.743700,30.271129,73.635972
833,834,Yellow Rose Country and Western Bar,-94.642420,30.146324,236.338997
907,908,Erwin Events Center,-97.733269,30.276865,74.331665


We do, unfortunately, have some bad data in the table. In particular, Jack's Bar and the Yellow Rose Country and Western Bar are not correctly geolocated. Seven Oaks should be inspected also.

In [91]:
venues_with_locations.iloc[322]

id                          323
name                 Jack's Bar
address      2950 Thousand Oaks
city                San Antonio
state                     Texas
zip                       78247
longitude            -98.441238
latitude              29.577499
Name: 322, dtype: object

Jack's is easily enough located on a map. I'll add the coordinates manually. It looks like a simple case of a missing negative sign in front of the longitude.

In [92]:
venues_with_locations.loc[venues_with_locations['id'] == 323, ['longitude', 'latitude']] = [-98.44123786548818, 29.577498688624814]

In [93]:
venues_with_locations.loc[venues_with_locations['id'] == 323]

,id,name,address,city,state,zip,longitude,latitude
322,323,Jack's Bar,2950 Thousand Oaks,San Antonio,Texas,78247,-98.441238,29.577499


With that fixed, I'll move on to the Yellow Rose.

In [94]:
venues_with_locations.loc[venues_with_locations['id'] == 834]

,id,name,address,city,state,zip,longitude,latitude
833,834,Yellow Rose Country and Western Bar,I10 and Wurzbach,San Antonio,Texas,78230,-94.64242,30.146324


The Yellow Rose was apparently at I-10 and Wurzbach. I can't be exactly sure of what corner it was on, but at least we have an approximate location. I'll pick the corner where the Colonnade is now, and maybe in the future we can have more accurate information. This will be much more precise than what we currently have.

In [95]:
venues_with_locations.loc[venues_with_locations['id'] == 834, ['longitude', 'latitude']] = [-98.56179835762765, 29.532788840280286]

In [96]:
venues_with_locations.loc[venues_with_locations['id'] == 834]

,id,name,address,city,state,zip,longitude,latitude
833,834,Yellow Rose Country and Western Bar,I10 and Wurzbach,San Antonio,Texas,78230,-98.561798,29.532789


Finally, I'll investigate the Seven Oaks. I find that it is, or was, on Austin Highway in San Antonio, so definitely not 73 miles away. I'll replace these coordinates with new ones.

In [97]:
venues_with_locations.loc[venues_with_locations['id'] == 615]

,id,name,address,city,state,zip,longitude,latitude
614,615,Seven Oaks,1400 Austin HWY,San Antonio,Texas,78209,-97.7437,30.271129


In [98]:
venues_with_locations.loc[venues_with_locations['id'] == 615, ['longitude', 'latitude']] = [-98.43887907258517, 29.491514899097037]

In [99]:
venues_with_locations.loc[venues_with_locations['id'] == 615]

,id,name,address,city,state,zip,longitude,latitude
614,615,Seven Oaks,1400 Austin HWY,San Antonio,Texas,78209,-98.438879,29.491515


## Write the Geolocation Data to the Excel File

Now that those major data issues are cleaned up, I'll write the data frame to our Excel file.

In [82]:
def write_df_to_excel(df, filepath, sheet_name):
    """Write a DataFrame to a sheet in an Excel file, creating the file
    if it doesn't exist, or replacing the sheet if it does."""
    
    mode = "a" if os.path.exists(filepath) else "w"
    kwargs = {"if_sheet_exists": "replace"} if mode == "a" else {}

    with pd.ExcelWriter(filepath, engine="openpyxl", datetime_format='YYYY-MM-DD', mode=mode, **kwargs) as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)

In [101]:
# write_df_to_excel(venues_with_locations, 'output-data/sounds-of-san-anto.xlsx', 'locations')